<a href="https://colab.research.google.com/github/haru452/anime_super-resolution/blob/main/smartohone_image_cutting_and_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Google ドライブのマウント

まず、Google ColabからGoogleドライブのファイルにアクセスできるように、ドライブをマウントします。実行すると認証を求められますので、指示に従ってください。

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Googleドライブ内のフォルダーから画像を取得

次に、特定のGoogleドライブフォルダーから画像ファイルの一覧を取得し、ダウンロードする方法を説明します。フォルダーのパスは、Googleドライブ上で確認できます。

**注意点:**
- `your_image_folder_path` を、実際に画像が保存されているGoogleドライブのフォルダーのパスに置き換えてください。
- `output_folder_path` を、処理した画像を保存したい新しいフォルダーのパスに置き換えてください。もし存在しない場合は、コードで作成されます。

In [3]:
import os
import shutil
from PIL import Image

# 画像が保存されているGoogleドライブのフォルダーのパス
your_image_folder_path = '/content/drive/MyDrive/research/Train/Train high'
# 処理した画像を保存する新しいフォルダーのパス
output_folder_path = '/content/drive/MyDrive/research/EX for YOLO'

# 出力フォルダーが存在しない場合は作成
os.makedirs(output_folder_path, exist_ok=True)

image_files = []
for root, _, files in os.walk(your_image_folder_path):
    for file in files:
        # 画像ファイルのみを対象とする (例: .jpg, .png)
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            image_files.append(os.path.join(root, file))

print(f"見つかった画像ファイル数: {len(image_files)}")
if image_files:
    print("最初の5つのファイル:")
    for f in image_files[:5]:
        print(f)

見つかった画像ファイル数: 313
最初の5つのファイル:
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260630-212845.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260630-212842.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260630-082858.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260627-124445.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260627-124655.png


## 画像処理：黒い部分の切り取りと削除

スマホのスクリーンショットで「イラストでない黒いところを切り取ってそれを削除する」というご要望ですが、これが具体的にどのような「黒いところ」を指しているかによって、処理の複雑さが変わります。

**一般的なアプローチ:**
1.  **均一な黒い枠の除去:** 画像の端に均一な黒い枠がある場合、それを自動的に検出してクロップ（切り取り）します。
2.  **特定の色の背景除去:** ある色が完全に背景として扱える場合、その色を透明にするなどの処理を行います。
3.  **オブジェクト検出/セグメンテーション:** 画像内の「イラスト」を特定し、それ以外の部分（黒い背景を含む）を削除する。これは高度な画像処理技術（例: 機械学習モデル）を必要とします。

ここでは、最も一般的なケースとして、**画像の端にある均一な黒い（または非常に暗い）枠を自動的にクロップする**方法を例として示します。もし、もっと複雑な背景の除去が必要な場合は、具体的な画像の例や「イラストでない黒いところ」の定義を詳しく教えていただけると、より適切な方法を提案できます。

以下のコードは、画像を読み込み、端の黒いピクセルを自動的にトリミングし、新しいフォルダーに保存するものです。`trim_black_borders` 関数がその処理を行います。

In [4]:
def process_screenshot(image_path, dark_threshold=10, content_pixel_ratio=0.1, fixed_top_crop_pixels=0, fixed_bottom_crop_pixels=0):
    """
    スマホのスクリーンショットを処理し、指定されたUI要素と暗い縁をトリミングします。
    fixed_top_crop_pixels: 上から強制的に切り取るピクセル数（ステータスバーなど）。
    fixed_bottom_crop_pixels: 下から強制的に切り取るピクセル数（ナビゲーションバーなど）。
    dark_threshold: 暗いとみなす明るさのしきい値 (0-255)。この値より小さい平均輝度を持つピクセルを暗いとみなします。
    content_pixel_ratio: 行/列が「内容物」とみなされるために必要な、dark_thresholdより明るいピクセルの最小比率。
    """
    try:
        img = Image.open(image_path).convert("RGB")
        width, height = img.size

        # 1. 固定ピクセルによる切り取り（ステータスバー、ナビゲーションバー除去）
        crop_top = fixed_top_crop_pixels
        crop_bottom = height - fixed_bottom_crop_pixels

        if crop_top >= crop_bottom or crop_top < 0 or crop_bottom > height:
            print(f"警告: {os.path.basename(image_path)}: 固定クロップの範囲が不正です。元の画像を返します。")
            return img

        img = img.crop((0, crop_top, width, crop_bottom))

        # 切り取った後の画像サイズで再計算
        pixels = img.load()
        width, height = img.size

        # helper function to check if a pixel is 'bright enough' (not dark)
        def is_bright_pixel(r, g, b):
            return (r + g + b) / 3 > dark_threshold

        # 2. 端の暗い部分のトリミング（これまでのロジック）
        top = 0
        for y in range(height):
            bright_pixels_in_row = 0
            for x in range(width):
                r, g, b = pixels[x, y]
                if is_bright_pixel(r, g, b):
                    bright_pixels_in_row += 1
            if bright_pixels_in_row / width >= content_pixel_ratio:
                top = y
                break

        bottom = height - 1
        for y in range(height - 1, -1, -1):
            bright_pixels_in_row = 0
            for x in range(width):
                r, g, b = pixels[x, y]
                if is_bright_pixel(r, g, b):
                    bright_pixels_in_row += 1
            if bright_pixels_in_row / width >= content_pixel_ratio:
                bottom = y
                break

        left = 0
        for x in range(width):
            bright_pixels_in_col = 0
            for y in range(height):
                r, g, b = pixels[x, y]
                if is_bright_pixel(r, g, b):
                    bright_pixels_in_col += 1
            if bright_pixels_in_col / height >= content_pixel_ratio:
                left = x
                break

        right = width - 1
        for x in range(width - 1, -1, -1):
            bright_pixels_in_col = 0
            for y in range(height):
                r, g, b = pixels[x, y]
                if is_bright_pixel(r, g, b):
                    bright_pixels_in_col += 1
            if bright_pixels_in_col / height >= content_pixel_ratio:
                right = x
                break

        # Validate crop area
        if right <= left or bottom <= top:
            print(f"警告: {os.path.basename(image_path)}: 内容物が見つからなかったか、無効なトリミング範囲です。固定クロップ後の画像を返します。")
            return img

        # 画像をクロップ
        cropped_img = img.crop((left, top, right + 1, bottom + 1))
        return cropped_img

    except Exception as e:
        print(f"エラー: {os.path.basename(image_path)} の処理中にエラーが発生しました: {e}")
        return None

# 全ての画像ファイルに対して処理を実行
processed_count = 0
for img_path in image_files:
    # fixed_top_crop_pixels と fixed_bottom_crop_pixels を調整してください。
    # 例: スマホのステータスバーが上から80ピクセル、ナビゲーションバーが下から120ピクセル程度の場合
    processed_img = process_screenshot(
        img_path,
        dark_threshold=10,            # 暗いとみなす明るさのしきい値
        content_pixel_ratio=0.1,      # コンテンツとみなす明るいピクセルの比率
        fixed_top_crop_pixels=80,     # 上から強制的に切り取るピクセル数
        fixed_bottom_crop_pixels=120  # 下から強制的に切り取るピクセル数
    )

    if processed_img:
        # 新しいファイルパスを生成
        file_name = os.path.basename(img_path)
        output_img_path = os.path.join(output_folder_path, file_name)

        # 画像を保存
        processed_img.save(output_img_path)
        processed_count += 1
        print(f"処理済み: {file_name} -> {output_img_path}")

print(f"\n合計 {processed_count} 個の画像を処理し、'{output_folder_path}' に保存しました。")

処理済み: Screenshot_20260630-212845.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260630-212845.png
処理済み: Screenshot_20260630-212842.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260630-212842.png
処理済み: Screenshot_20260630-082858.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260630-082858.png
処理済み: Screenshot_20260627-124445.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260627-124445.png
処理済み: Screenshot_20260627-124655.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260627-124655.png
処理済み: Screenshot_20260630-212920.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260630-212920.png
処理済み: Screenshot_20260626-213850.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260626-213850.png
処理済み: Screenshot_20260626-213926.png -> /content/drive/MyDrive/research/EX for YOLO/Screenshot_20260626-213926.png
処理済み: Screenshot_20260627-124750.png -> /content/drive/MyDrive/research/EX for Y

##実験

In [5]:


# 画像が保存されているGoogleドライブのフォルダーのパス
your_image_folder_path =  '/content/drive/MyDrive/research/Train/Train high'
# 処理した画像を保存する新しいフォルダーのパス

image_files = []
for root, _, files in os.walk(your_image_folder_path):
    for file in files:
        # 画像ファイルのみを対象とする (例: .jpg, .png)
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            image_files.append(os.path.join(root, file))

print(f"見つかった画像ファイル数: {len(image_files)}")
if image_files:
    print("最初の5つのファイル:")
    for f in image_files[:5]:
        print(f)

見つかった画像ファイル数: 313
最初の5つのファイル:
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260630-212845.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260630-212842.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260630-082858.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260627-124445.png
/content/drive/MyDrive/research/Train/Train high/Screenshot_20260627-124655.png


In [6]:
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00


In [8]:
import os
import cv2
import shutil # コピー用に追加
from PIL import Image
from ultralytics import YOLO

# 1. 出力先フォルダーの準備
yolo_output_dir = '/content/drive/MyDrive/Research/YOLO_Output'
os.makedirs(yolo_output_dir, exist_ok=True)

# 2. モデルの読み込み
model = YOLO('yolov8n.pt')
MODE = 'detection'
processed_count = 0
not_detected_count = 0 # 検出なしカウント用

for img_path in image_files:
    file_name = os.path.basename(img_path)
    name, ext = os.path.splitext(file_name)

    try:
        # YOLOで画像を推論
        results = model(img_path, conf=0.10, verbose=False)
        result = results[0]

        # --- 変更点: 検出オブジェクトがない場合 ---
        if len(result.boxes) == 0:
            # そのままコピーして保存
            save_path = os.path.join(yolo_output_dir, f"{name}_original{ext}")
            shutil.copy(img_path, save_path)
            print(f"検出なし（そのまま保存）: {file_name}")
            not_detected_count += 1
            continue

        # 元画像をOpenCVで読み込み
        img = cv2.imread(img_path)
        if img is None:
            continue

        # 1つの画像に複数キャラがいる場合を考慮してループ
        # (以下、検出された場合の処理はそのまま)
        for i, box in enumerate(result.boxes):
            class_id = int(box.cls[0])
            if class_id != 0:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            if MODE == 'detection':
                cropped = img[y1:y2, x1:x2]
            # (セグメンテーションの処理は省略していますが、そのまま使えます)
            else:
                cropped = img[y1:y2, x1:x2]

            output_name = f"{name}_{i}.png"
            save_path = os.path.join(yolo_output_dir, output_name)

            cv2.imwrite(save_path, cropped)
            print(f"保存完了: {output_name}")
            processed_count += 1

    except Exception as e:
        print(f"エラー（{file_name}）: {e}")

print(f"\nすべて完了！")
print(f"検出・切り出し保存数: {processed_count} 個")
print(f"検出なしでそのまま保存した数: {not_detected_count} 個")
print(f"保存先: '{yolo_output_dir}'")

保存完了: Screenshot_20260627-124650_1.png
保存完了: Screenshot_20260627-124650_2.png
保存完了: Screenshot_20260627-124929_0.png
保存完了: Screenshot_20260627-124701_0.png
保存完了: Screenshot_20260626-213900_0.png
保存完了: Screenshot_20260627-124812_0.png
保存完了: Screenshot_20260627-124812_1.png
保存完了: Screenshot_20260627-124330_0.png
保存完了: Screenshot_20260701-064604_0.png
保存完了: Screenshot_20260627-123624_1.png
保存完了: Screenshot_20260701-064607_0.png
保存完了: Screenshot_20260627-124057_0.png
保存完了: Screenshot_20260627-124057_3.png
保存完了: Screenshot_20260627-124104_1.png
保存完了: Screenshot_20260627-124122_0.png
保存完了: Screenshot_20260626-213919_0.png
検出なし（そのまま保存）: Screenshot_20260626-214750.png
保存完了: Screenshot_20260626-213906_7.png
保存完了: Screenshot_20260701-064601_1.png
保存完了: Screenshot_20260627-125253_7.png
保存完了: Screenshot_20260627-124532_0.png
保存完了: Screenshot_20260627-124609_0.png
保存完了: Screenshot_20260627-124718_0.png
保存完了: Screenshot_20260627-125450_0.png
保存完了: Screenshot_20260627-124613_0.png
保存完了: Screenshot_20

In [9]:
# 画像が保存されているGoogleドライブのフォルダーのパス
your_image_folder_path ='/content/drive/MyDrive/Research/YOLO_Output'

image_files = []
for root, _, files in os.walk(your_image_folder_path):
    for file in files:
        # 画像ファイルのみを対象とする (例: .jpg, .png)
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            image_files.append(os.path.join(root, file))

print(f"見つかった画像ファイル数: {len(image_files)}")

見つかった画像ファイル数: 321
